## Reinforcement Finetuning of Retail Agent

Finetune a Retail Agent with Reinforcement learning


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

## Validate Grader

In [23]:
import json
import os
import requests

with open("tool_call_grader/tool_call_grader.py", "r") as f:
    grader_source = f.read()

grader = {
    "type": "python",
    "source": grader_source,
    "pass_threshold": 0.5,
}

headers = {
    "Authorization": f"Bearer {os.environ['AZURE_OPENAI_API_KEY']}",
}

validate_response = requests.post(
    os.environ['AZURE_OPENAI_ENDPOINT'] + "/openai/v1/fine_tuning/alpha/graders/validate",
    headers=headers,
    json={
        "grader": grader,
    },
)

print(f"Status: {validate_response.status_code}")
print(json.dumps(validate_response.json(), indent=2))

Status: 200
{
  "grader": {
    "type": "python",
    "source": "import json\nfrom collections import Counter\nfrom typing import Any, Dict, List\n\n\ndef grade(sample: Dict, item: Dict) -> float:\n    actual_calls = sample.get(\"output_tools\") or []\n    if not actual_calls:\n        try:\n            # load from output_text to make it work with /graders/run API which doesn't support output_tools yet\n            actual_calls = json.loads(sample[\"output_text\"])[\"tool_calls\"]\n        except (KeyError, json.JSONDecodeError):\n            pass\n\n    expected_calls = item[\"expected_tool_calls\"]\n    return grade_tool_calls(actual_calls, expected_calls)\n\n\ndef grade_tool_calls(actual: List[Dict], expected: List[Dict]) -> float:\n    if not expected and not actual:\n        return 1.0\n    if not expected or not actual:\n        return 0.0\n\n    actual_names = [c[\"function\"][\"name\"] for c in actual]\n    expected_names = [c[\"function\"][\"name\"] for c in expected]\n\n    #

In [26]:
sample_model_output = {
    "role": "assistant",
    "tool_calls": [
        {
            "id": "call_1",
            "type": "function",
            "function": {
                "name": "get_order_status",
                "arguments": json.dumps({"order_id": "123"}),
            },
        }
    ],
}

run_payload = {
    "grader": grader,
    "model_sample": json.dumps(sample_model_output),
    "item": {
        "expected_tool_calls": [
            {
                "id": "call_1",
                "type": "function",
                "function": {
                    "name": "get_order_status",
                    "arguments": json.dumps({"order_id": "321"}),
                },
            },
        ],
    },
}

run_response = requests.post(
    os.environ['AZURE_OPENAI_ENDPOINT'] + "/openai/v1/fine_tuning/alpha/graders/run",
    headers=headers,
    json=run_payload,
)

print(f"Status: {run_response.status_code}")
print(run_response.json())

Status: 200
{'reward': 0.5, 'metadata': {'name': 'grader-1BR2vT5gPyve', 'type': 'python', 'errors': {'formula_parse_error': False, 'sample_parse_error': False, 'sample_parse_error_details': None, 'truncated_observation_error': False, 'unresponsive_reward_error': False, 'invalid_variable_error': False, 'invalid_variable_error_details': None, 'other_error': False, 'python_grader_server_error': False, 'python_grader_server_error_type': None, 'python_grader_runtime_error': False, 'python_grader_runtime_error_details': None, 'model_grader_server_error': False, 'model_grader_refusal_error': False, 'model_grader_refusal_error_details': None, 'model_grader_parse_error': False, 'model_grader_parse_error_details': None, 'model_grader_exceeded_max_tokens_error': False, 'model_grader_server_error_details': None, 'endpoint_grader_internal_error': False, 'endpoint_grader_internal_error_details': None, 'endpoint_grader_server_error': False, 'endpoint_grader_server_error_details': None, 'endpoint_grad